# 04 · AGE vs SQL vs NetworkX — The Arena

The honest three-way comparison on the same question (variable-length reachability),
scored on three axes:

- **Readability** — lines of code per approach.
- **Correctness** — all three agree with each other and with planted truth.
- **Runtime** — how cost grows with hop depth.

No single winner: AGE and NetworkX express traversal cleanly while SQL balloons; NetworkX
is fastest in memory but must load the whole graph into Python; AGE runs in the database
next to the data.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt

from src.age.connect import age_available, get_connection
from src.baselines.sql_recursive import _REACHABLE
from src.config import load_settings
from src.datasets.graphs import KNOWS, PAYS, all_graphs
from src.evaluation.comparison import Implementation, compare
from src.evaluation.correctness import all_rings_found, sets_agree
from src.evaluation.readability import count_loc
from src.evaluation.runtime import benchmark
from src.outputs import figure_path, save_json
from src.queries import traversal as tv
from src.queries.rings import find_rings_networkx, find_rings_sql
from src.queries.traversal import variable_length_cypher

settings = load_settings()
AGE = age_available(settings)
social = all_graphs(settings)["social_txn"]
conn = get_connection(settings) if AGE else None
START = 0
GRAPH = settings.graphs.social
print("AGE:", AGE)

AGE: True


## 1. Readability — lines of code

Same answer, three sizes. The SQL is the real recursive CTE from the project; the Cypher
is one line.

In [2]:
nx_lines = [
    "result, stack = set(), [(start, 0, frozenset({start}))]",
    "while stack:",
    "    node, depth, seen = stack.pop()",
    "    if min_hops <= depth <= max_hops: result.add(node)",
    "    if depth >= max_hops: continue",
    "    for nbr in G.successors(node):",
    "        if nbr not in seen: stack.append((nbr, depth + 1, seen | {nbr}))",
]
loc = {
    "AGE (Cypher)": count_loc(variable_length_cypher(START, 1, 6, KNOWS)),
    "SQL (recursive CTE)": count_loc(_REACHABLE),
    "NetworkX": count_loc("\n".join(nx_lines)),
}
print(loc)
save_json("04_loc.json", loc)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.barh(list(loc), list(loc.values()), color=["#4c6ef5", "#e4572e", "#2f9e44"])
ax.set_xlabel("lines of code (same reachability question)")
ax.set_title("Readability: SQL balloons, Cypher is one line")
for i, v in enumerate(loc.values()):
    ax.text(v + 0.1, i, str(v), va="center")
fig.tight_layout()
fig.savefig(figure_path("04_loc.png"), dpi=120)
plt.close(fig)

{'AGE (Cypher)': 1, 'SQL (recursive CTE)': 11, 'NetworkX': 7}


## 2. Correctness — everyone agrees, and matches planted truth

Reachability sets agree across approaches at every depth (comparing without the start
account, per the openCypher walk-vs-simple-path note in notebook 02), and all three find
exactly the planted fraud rings.

In [3]:
depths = [1, 2, 3, 4, 5]
agree_by_depth = {}
for d in depths:
    nx_set = tv.reachable_networkx(social, START, 1, d, KNOWS)
    sql_set = tv.reachable_sql(social, START, 1, d, KNOWS)
    age_set = (
        tv.reachable_age(conn, GRAPH, START, 1, d, KNOWS) - {START} if AGE else set(nx_set)
    )
    agree_by_depth[d] = sets_agree(nx_set, sql_set, age_set)
    assert agree_by_depth[d]

rings_ok = all_rings_found(find_rings_networkx(social, 3, 5, PAYS), social.rings)
print("reachability agreement by depth:", agree_by_depth)
print("all planted rings found:", rings_ok)
save_json("04_correctness.json", {
    "reachability_agree_by_depth": agree_by_depth,
    "all_approaches_agree": all(agree_by_depth.values()),
    "all_planted_rings_found": rings_ok,
})

reachability agreement by depth: {1: True, 2: True, 3: True, 4: True, 5: True}
all planted rings found: True


PosixPath('/workspaces/apache_age_101/outputs/data/04_correctness.json')

## 3. Runtime — how cost grows with depth

Median of three runs per depth. SQL enumerates simple paths, so its cost climbs with
depth; AGE and NetworkX stay cheap. (Log scale.)

In [4]:
runtime = {"AGE": [], "SQL": [], "NetworkX": []}
for d in depths:
    runtime["NetworkX"].append(benchmark(tv.reachable_networkx, social, START, 1, d, KNOWS, repeat=3))
    runtime["SQL"].append(benchmark(tv.reachable_sql, social, START, 1, d, KNOWS, repeat=3))
    if AGE:
        runtime["AGE"].append(
            benchmark(tv.reachable_age, conn, GRAPH, START, 1, d, KNOWS, repeat=3)
        )

if not AGE:
    runtime.pop("AGE")

fig, ax = plt.subplots(figsize=(7, 4))
colours = {"AGE": "#4c6ef5", "SQL": "#e4572e", "NetworkX": "#2f9e44"}
for name, series in runtime.items():
    ax.plot(depths, [s * 1000 for s in series], marker="o", label=name, color=colours[name])
ax.set_xlabel("hop depth")
ax.set_ylabel("milliseconds (median of 3)")
ax.set_yscale("log")
ax.set_title("Runtime vs depth")
ax.legend()
ax.grid(True, alpha=0.3, which="both")
fig.tight_layout()
fig.savefig(figure_path("04_runtime_vs_depth.png"), dpi=120)
plt.close(fig)
save_json("04_runtime_vs_depth.json",
          {"depths": depths, "milliseconds": {k: [s * 1000 for s in v] for k, v in runtime.items()}})
print("saved runtime figure + json")

saved runtime figure + json


## 4. The scoreboard

One representative question (reachable within 4 hops), every approach, with lines of code,
runtime, and a correctness flag against planted truth.

In [5]:
DEPTH = 4
truth = tv.reachable_networkx(social, START, 1, DEPTH, KNOWS)
impls = [
    Implementation(
        "AGE (Cypher)",
        lambda: (tv.reachable_age(conn, GRAPH, START, 1, DEPTH, KNOWS) - {START})
        if AGE else set(truth),
        variable_length_cypher(START, 1, DEPTH, KNOWS),
    ),
    Implementation(
        "SQL (recursive CTE)",
        lambda: tv.reachable_sql(social, START, 1, DEPTH, KNOWS),
        _REACHABLE,
    ),
    Implementation(
        "NetworkX",
        lambda: tv.reachable_networkx(social, START, 1, DEPTH, KNOWS),
        "\n".join(nx_lines),
    ),
]
board = compare(impls, truth)
save_json("04_arena.json", board.to_dict(orient="records"))
if conn is not None:
    conn.close()
board

,approach,lines_of_code,seconds,correct
0,AGE (Cypher),1,0.298239,True
1,SQL (recursive CTE),11,0.010968,True
2,NetworkX,7,0.005586,True


**Takeaway.** Each approach owns a region. Cypher through AGE wins readability and
runs next to your data; NetworkX is fastest once the graph is in memory; SQL works but its
readability and runtime degrade with depth. The right answer depends on where your data
lives and what you need — which is exactly the decision framework in notebook 06.